# CoAD reproduction — setup, data, EDA

Generated from `huong-dan-reproduce-coad.md`, sections 16–21. Run top to bottom.
Full logs are written to Drive; cells print only short tails.

**Drive folder:** `STORE = MyDrive/CoAD_project`. If your folder lives elsewhere (or is shared with you: add a shortcut to My Drive first), edit `STORE` in cell 16.2.


## 16.1. Create the notebook and check resources

Select a GPU runtime (T4) for the CoAD part. Prints Python, OS and GPU.


In [ ]:
import sys, platform, subprocess
print('Python:', sys.version)
print('OS:', platform.platform())
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=False,
)
print('GPU:', gpu.stdout.strip() or gpu.stderr.strip())
print('GPU query exit code:', gpu.returncode)

## 16.2. Mount Drive and create the storage layout

When asked, authorize the Google account that **owns** `CoAD_project`; otherwise an empty folder is created in the wrong account. The second cell defines `tail`, `run_cmd`, `grep_file`: full logs go to Drive, cells print only short tails. After a session restart, re-run both cells plus 16.3.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
STORE = Path('/content/drive/MyDrive/CoAD_project')
for folder in ['env', 'data_source', 'eda', 'runs', 'reports']:
    (STORE / folder).mkdir(parents=True, exist_ok=True)
print(STORE)

In [ ]:
import subprocess, sys

def tail(text, n=15):
    """Print only the last n lines; the full text is kept in the log file."""
    lines = text.strip().splitlines()
    if len(lines) > n:
        print(f'... ({len(lines) - n} earlier lines omitted)')
    print('\n'.join(lines[-n:]))

def run_cmd(cmd, log_path, n=10, **kwargs):
    """Run a command, write the full stdout+stderr to log_path, print a short tail."""
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    log = result.stdout + '\n' + result.stderr
    Path(log_path).write_text(log, encoding='utf-8')
    print('Exit code:', result.returncode, '| full log:', log_path)
    tail(log, n if result.returncode == 0 else 40)
    return result

def grep_file(path, pattern, context=0):
    """Print only the lines of a source file that match the regex pattern, with line numbers."""
    import re
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    hits = [i for i, line in enumerate(lines) if re.search(pattern, line)]
    shown = sorted({j for i in hits for j in range(max(0, i - context), min(len(lines), i + context + 1))})
    for j in shown:
        print(f'{j + 1:4d}: {lines[j]}')
    print(f'-- {len(hits)} matching lines in {Path(path).name}')

In [ ]:
# Check that the right Drive folder is mounted: expect data_source, eda, env, reports, runs
print(sorted(p.name for p in STORE.iterdir()))
print('commit.txt exists:', (STORE / 'env' / 'commit.txt').exists())

## 16.3. Clone, save the commit, list dependencies

Stops if the clone differs from the commit saved in `env/commit.txt`.


In [ ]:
REPO = Path('/content/CoAD')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
commit_file = STORE / 'env' / 'commit.txt'
if commit_file.exists():
    saved_commit = commit_file.read_text(encoding='utf-8').strip()
    assert commit == saved_commit, (
        f'Current commit {commit} differs from saved commit {saved_commit}. '
        'Inspect local changes and restore the expected commit before continuing.'
    )
else:
    commit_file.write_text(commit, encoding='utf-8')
print('Commit:', commit)
reqs = [l.strip() for l in (REPO / 'requirements.txt').read_text().splitlines()
        if l.strip() and not l.startswith('#')]
print(f'{len(reqs)} requirements:', ', '.join(reqs))

## 16.4. Dry-run before installing

Already done (exit code 0). Skip unless the runtime or requirements changed.


In [ ]:
result = run_cmd(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', '-r', str(REPO / 'requirements.txt')],
    STORE / 'env' / 'dependency-dry-run.log', n=3,
)
assert result.returncode == 0, 'Read the error log before installing packages'

## 16.5. Install and check dependencies

Skip if packages are still installed in this runtime. If Colab asks to restart the session, restart and re-run 16.2–16.3, not this cell.


In [ ]:
install = run_cmd(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')],
    STORE / 'env' / 'install.log', n=2,
)
assert install.returncode == 0, 'Installation failed; inspect install.log'

dependency_check = run_cmd(
    [sys.executable, '-m', 'pip', 'check'], STORE / 'env' / 'pip-check.log', n=20,
)

## 16.6. Import CoAD and run a real GPU computation

Expected: `Import CoAD: OK`, `CUDA available: True`, `GPU computation: OK`, `Exit code: 0`.


In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path('/content/CoAD')
STORE = Path('/content/drive/MyDrive/CoAD_project')

code = """
import sys
# evaluation/evaluator.py appends sys.path[0] + '/evaluation'; with `python -c`, sys.path[0] is ''
# so put the repo first, as when running `python train_classify_mask.py`.
sys.path.insert(0, '/content/CoAD')
import torch
from toolkit.utils import load_dataset
from evaluation.evaluator import evaluate
from model.patch_soft_mask import PatchFrequencyMask

print('Import CoAD: OK')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU is not available'
print('GPU:', torch.cuda.get_device_name(0))
x = torch.randn(128, 128, device='cuda')
z = x @ x
torch.cuda.synchronize()
assert torch.isfinite(z).all().item(), 'GPU computation produced non-finite values'
print('GPU computation: OK')
"""
result = subprocess.run(
    [sys.executable, '-c', code], cwd=REPO,
    capture_output=True, text=True,
)
test_log = result.stdout + '\n' + result.stderr
(STORE / 'env' / 'import-gpu-check.log').write_text(test_log, encoding='utf-8')
print(test_log)
print('Exit code:', result.returncode)
assert result.returncode == 0, 'Read the traceback before proceeding to the dataset'

## 16.7. Save an environment snapshot

Creates a timestamped folder under `env/` with pip freeze, nvidia-smi and environment.json.


In [ ]:
import json, platform
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
snapshot = STORE / 'env' / stamp
snapshot.mkdir(parents=True, exist_ok=False)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(snapshot / 'pip-freeze.txt').write_text(freeze, encoding='utf-8')
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
(snapshot / 'nvidia-smi.txt').write_text(gpu_info.stdout + gpu_info.stderr, encoding='utf-8')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
info = {
    'python': sys.version, 'os': platform.platform(), 'commit': commit,
    'purpose': 'Python 3.13 sanity-check environment; repo recommends 3.11',
    'full_pipeline_verified': False,
}
(snapshot / 'environment.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
print('Environment snapshot:', snapshot)

## 17.1. Download, verify and extract TSB-AD-U

The author's dataset link (README) has been deleted, so the official TSB-AD-U archive is used. The archive is cached in `data_source/`; later runs only extract. Expected: 870 CSV files and `236 -> 236_SED_id_3_Medical_tr_3740_1st_3840.csv`. Group IDs are positions in the sorted file list, so `raw/` must contain exactly these 870 files.


In [ ]:
import hashlib, shutil, urllib.request, zipfile
from datetime import date

URL = 'https://www.thedatum.org/datasets/TSB-AD-U.zip'
EXPECTED_SHA256 = '0c47020d3423723c70773736dbd800369f2b487328becbf339450d1ae5020961'
archive = STORE / 'data_source' / 'TSB-AD-U.zip'

if not archive.exists():
    tmp = Path('/content/TSB-AD-U.zip')
    # The server returns 403 to the default 'Python-urllib' User-Agent.
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, tmp.open('wb') as out:
        shutil.copyfileobj(resp, out)
    shutil.copy(tmp, archive)
    (STORE / 'data_source' / 'TSB-AD-U.source.txt').write_text(
        f'url={URL}\ndownloaded={date.today()}\n', encoding='utf-8')

sha = hashlib.sha256(archive.read_bytes()).hexdigest()
print('Archive:', archive, f'{archive.stat().st_size:,} bytes')
print('SHA-256 matches the value checked on 2026-10-12:', sha == EXPECTED_SHA256)
# A different hash means the source was updated: record it, do not ignore it.

with zipfile.ZipFile(archive) as z:
    members = [m for m in z.namelist() if not m.endswith('/')]
    bad = [m for m in members
           if m.startswith('/') or '..' in m or not m.startswith('TSB-AD-U/') or not m.endswith('.csv')]
    assert not bad, f'Unexpected entries in archive: {bad[:5]}'
    print('CSV files in archive:', len(members))

    RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
    RAW.mkdir(parents=True, exist_ok=True)
    for m in members:                       # flatten TSB-AD-U/xxx.csv -> raw/xxx.csv
        target = RAW / Path(m).name
        if not target.exists():
            with z.open(m) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)

files = sorted(p.name for p in RAW.iterdir())
assert len(files) == 870 and all(f.endswith('.csv') for f in files), \
    'raw/ must contain exactly the 870 CSVs: group IDs are positions in the sorted listing'
for g in [225, 236, 237, 256, 260, 276, 287, 301]:
    print(g, '->', files[g - 1])

## 17.2. Read the loader (only the relevant lines)

TSB-AD branch: train = first N points (N from `_tr_N_` in the file name), test = the full series, min-max fitted on the train segment, subsequence length estimated from the data.


In [ ]:
import inspect, os
os.chdir(REPO)                        # load_dataset uses paths relative to the repo
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from toolkit.utils import load_dataset
print(inspect.signature(load_dataset))
src = inspect.getsource(load_dataset)
start = src.index("elif data_name == \"TSB-AD\"")
print(src[start:src.index('elif data_name == "ASD"')])   # TSB-AD branch only

## 17.3. Call the loader for one group

Pilot group 236, chosen before seeing any result. Expected: train `(3740,)`, test/label `(39999,)`, labels `[0 1]`, 1472 anomaly points. Data are **after the loader preprocessing**.


In [ ]:
import numpy as np
DATASET = 'TSB-AD'
GROUP = '236'          # chosen in 17.0 before seeing any result
trains, tests, labels, periods = load_dataset(data_name=DATASET, group_name=[GROUP])
assert len(trains) == len(tests) == len(labels) == len(periods) == 1
train = np.asarray(trains[0])
test = np.asarray(tests[0])
y = np.asarray(labels[0]).reshape(-1)
SOURCE_FILE = sorted(os.listdir('dataset/TSB-AD/raw'))[int(GROUP) - 1]
print('File:', SOURCE_FILE)
print('Train/test/label:', train.shape, test.shape, y.shape)
print('Label values:', np.unique(y), '| anomaly points:', int(y.sum()))
print('Subsequence length:', periods[0])
print('Train range after loader min-max:', float(train.min()), float(train.max()))

## 17.4. Create the manifest

Add one row per series before comparing methods; never pick groups by their results.


In [ ]:
import pandas as pd
manifest = pd.DataFrame([{
    'dataset': DATASET, 'group': GROUP,
    'source_file': f'dataset/TSB-AD/raw/{SOURCE_FILE}',
    'data_version': f'TSB-AD-U.zip sha256={sha[:16]}..., source {URL}',
    'selection_reason': 'Pilot: short series present in both runners; chosen before running',
    'split_rule': 'train = first N points (N from _tr_N_ in file name); test = full series',
    'preprocessing': 'dropna; min-max fitted on train segment, applied to train and test',
    'deviation': 'Author dataset link deleted; official TSB-AD-U used instead',
    'commit': commit,
}])
manifest.to_csv(STORE / 'eda' / 'manifest.csv', index=False)
print(manifest.T)

## 18.1. Statistics for one series

Label imbalance and segment lengths; descriptive only.


In [ ]:
def as_2d(a):
    a = np.asarray(a)
    if a.ndim == 1:
        return a[:, None]
    if a.ndim != 2:
        raise ValueError(f'Expected time x channels, got {a.shape}')
    return a

def anomaly_segments(y):
    y = np.asarray(y).reshape(-1)
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Verified binary labels (0/1) are required')
    edges = np.diff(np.r_[0, y.astype(int), 0])
    starts = np.flatnonzero(edges == 1)
    ends = np.flatnonzero(edges == -1)  # exclusive
    return list(zip(starts, ends))

def describe_series(dataset, group, train, test, y):
    a, b = as_2d(train), as_2d(test)
    y = np.asarray(y).reshape(-1)
    assert len(b) == len(y), 'Test data and labels have different lengths'
    assert a.shape[1] == b.shape[1], 'Train and test data have different channel counts'
    assert len(a) > 0 and len(b) > 0
    segments = anomaly_segments(y)
    lengths = [end - start for start, end in segments]
    return {
        'dataset': dataset, 'group': str(group),
        'n_train': len(a), 'n_test': len(b), 'channels': a.shape[1],
        'train_nan': int(np.isnan(a).sum()), 'test_nan': int(np.isnan(b).sum()),
        'train_inf': int(np.isinf(a).sum()), 'test_inf': int(np.isinf(b).sum()),
        'anomaly_points': int(y.sum()), 'anomaly_ratio': float(y.mean()),
        'anomaly_segments': len(segments),
        'median_segment_length': float(np.median(lengths)) if lengths else 0,
        'max_segment_length': max(lengths, default=0),
    }

stats = describe_series(DATASET, GROUP, train, test, y)
display(pd.DataFrame([stats]))
pd.DataFrame([stats]).to_csv(STORE / 'eda' / f'{DATASET}_{GROUP}_stats.csv', index=False)

## 18.2. Check each channel and plot the signal

Red regions mark labeled anomalies. For TSB-AD the test plot covers the whole series, including the train segment.


In [ ]:
import matplotlib.pyplot as plt
a, b = as_2d(train), as_2d(test)
if not (np.isfinite(a).all() and np.isfinite(b).all()):
    raise ValueError('NaN/Inf detected: record the issue and inspect the loader before continuing')

summary = pd.concat({
    'train': pd.DataFrame(a).describe().T,
    'test': pd.DataFrame(b).describe().T,
}, names=['split', 'channel'])
summary.to_csv(STORE / 'eda' / f'{DATASET}_{GROUP}_distribution.csv')
display(summary)

channel = 0
fig, ax = plt.subplots(2, 1, figsize=(14, 6))
ax[0].plot(a[:, channel], linewidth=0.7)
ax[0].set_title(f'{DATASET}/{GROUP}: train, channel {channel}')
ax[1].plot(b[:, channel], linewidth=0.7)
for start, end in anomaly_segments(y):
    ax[1].axvspan(start, end, color='red', alpha=0.2)
ax[1].set_title('Test; red regions indicate labeled anomalies')
for axis in ax:
    axis.set_xlabel('Sample index within the split')
fig.tight_layout()
fig.savefig(STORE / 'eda' / f'{DATASET}_{GROUP}_signal.png', dpi=160)
plt.show()

## 18.3. Distribution, trend and periodicity

EDA only; does not replace the repo preprocessing.


In [ ]:
s = pd.Series(a[:, channel])
rolling_window = min(100, max(2, len(s) // 20))  # for EDA only
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(a[:, channel], bins=50, density=True, alpha=0.5, label='train')
ax[0].hist(b[:, channel], bins=50, density=True, alpha=0.5, label='test')
ax[0].legend(); ax[0].set_title('Train/test distributions')
ax[1].plot(s, alpha=0.4, linewidth=0.6)
ax[1].plot(s.rolling(rolling_window, min_periods=1).mean())
ax[1].set_title(f'Train + rolling mean ({rolling_window} samples)')
max_lag = min(200, len(s) // 4)
lags = range(1, max_lag + 1)
ax[2].plot(list(lags), [s.autocorr(lag=k) for k in lags])
ax[2].set_title('Training data autocorrelation')
ax[2].set_xlabel('Lag')
fig.tight_layout()
fig.savefig(STORE / 'eda' / f'{DATASET}_{GROUP}_patterns.png', dpi=160)
plt.show()

## 18.4. Data-quality table

Runs 8 checks (NaN/Inf, label length/values, anomalies in the train segment, constant signal, test values outside the train range, flat-line runs, 1-point anomaly segments). Saves `eda/<dataset>_<group>_quality.csv` and `reports/data_quality.md`; fill the `team_decision` column afterwards. Needs 18.1.


In [ ]:
def longest_constant_run(x):
    """Length of the longest run of identical consecutive values in a 1-D array."""
    same = (np.diff(x) == 0).astype(int)
    runs = [end - start for start, end in anomaly_segments(same)]
    return max(runs, default=0) + 1 if len(x) else 0

def quality_checks(dataset, group, train, test, y):
    a, b = as_2d(train), as_2d(test)
    y = np.asarray(y).reshape(-1)
    series = f'{dataset}/{group}'
    rows = []

    def add(issue, count, evidence, repo_handling):
        rows.append({'series': series, 'issue': issue, 'count': count, 'evidence': evidence,
                     'repo_handling': repo_handling, 'team_decision': 'TODO'})

    # 1. Missing / infinite values
    add('NaN/Inf', int((~np.isfinite(a)).sum() + (~np.isfinite(b)).sum()),
        'np.isfinite on train and test after the loader',
        'Loader calls dropna() on the raw CSV before splitting')

    # 2. Labels: length and values
    add('Label length differs from test length', abs(len(y) - len(b)),
        f'len(test)={len(b)}, len(labels)={len(y)}', 'Labels come from the same CSV rows as test')
    add('Non-binary label values', int((~np.isin(y, [0, 1])).sum()),
        f'unique labels: {np.unique(y).tolist()}', "Label column cast with astype(int)")

    # 3. Anomalies inside the training segment (only when test starts with the train segment)
    if len(b) >= len(a) and np.allclose(b[:len(a)], a, equal_nan=True):
        add('Labeled anomalies inside the training segment', int(y[:len(a)].sum()),
            f'sum(labels[:{len(a)}]); test starts with the train segment',
            'Train = first N points; no filtering of anomalies')

    for ch in range(a.shape[1]):
        tr, te = a[:, ch], b[:, ch]
        # 4. Constant channel -> min-max divides by zero
        add(f'Constant training signal (ch {ch})', int(np.ptp(tr) == 0),
            f'train min={tr.min():.4g}, max={tr.max():.4g}',
            'Min-max fitted on train; max == min would divide by zero')
        # 5. Values outside the training range
        outside = (te < tr.min()) | (te > tr.max())
        n_out = int(outside.sum())
        in_anom = int((outside & (y == 1)).sum()) if len(y) == len(te) else 'N/A'
        add(f'Test values outside the training range (ch {ch})', n_out,
            f'{in_anom} of them inside labeled anomalies; test range [{te.min():.3g}, {te.max():.3g}]',
            'Kept; min-max uses train statistics, so values may exceed [0, 1]')
        # 6. Flat-line: long runs of identical consecutive values
        run = longest_constant_run(te)
        add(f'Longest run of identical values (ch {ch})', run,
            'consecutive equal values in test', 'Kept; no deduplication')

    # 7. Very short anomaly segments
    lengths = [end - start for start, end in anomaly_segments(y)]
    add('Anomaly segments of length 1', sum(l == 1 for l in lengths),
        f'{len(lengths)} segments in total', 'Evaluated as given by the labels')
    return rows

def to_markdown_table(df):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for _, r in df.iterrows():
        lines.append('| ' + ' | '.join(str(r[c]).replace('|', '/') for c in cols) + ' |')
    return '\n'.join(lines)

quality = pd.DataFrame(quality_checks(DATASET, GROUP, train, test, y))
quality.to_csv(STORE / 'eda' / f'{DATASET}_{GROUP}_quality.csv', index=False)
report = STORE / 'reports' / 'data_quality.md'
report.write_text('# Data quality\n\n' + to_markdown_table(quality) + '\n', encoding='utf-8')
display(quality)
print('Saved:', report)

## 19.1. Preprocessing of group 236: expected values

Reference table (no code). Values computed from the repo code at commit `new` on the real data; cell 19.2 recomputes them. **If 19.2 disagrees, trust 19.2.**

| Step | Value for group 236 | Source in the repo |
|---|---|---|
| Train / test points | 3,740 / 39,999 | `_tr_3740_` in the file name; test = full series |
| Normalization | `(x - min_train) / (max_train - min_train)`, fitted on the train segment, applied to train and test; `max == min` not handled | `load_dataset`, TSB-AD branch |
| Period, `find_length_rank` | **125 = fallback default**: highest ACF peak at lag 335 (> 300 cap) | `find_length_rank`: `if ... > 300: return 125` |
| Period, `find_period` | 335 (ACF peak within 20-1000) | `find_period` |
| `subsequence_length` (`periods[0]`) | min(125, 335) = **125** | `load_dataset` |
| Window multiple | 4 (3740 // 125 = 29 > `window_multiple` 4) | `train_classify_mask.py` lines 84-87 |
| `window_length` | 125 x 4 = 500, rounded down to a multiple of `patch_size` 8 = **496** | lines 90-91 |
| Train stride | **6** (1000 <= 3740 < 10000) | lines 96-105 |
| Test stride | **125** (= `subsequence_length`) | `test_stride=subsequence_length` |
| Padding | zeros added at the **start**: train 2, test 122 | `sequence_to_windows` |
| Number of windows | train 542, test 318 | from the values above |
| Score merging | each test point is covered by up to 4 windows (min 1 at the edges); point score = **mean** over covering windows; padding removed, so score length = 39,999 = label length | `windows_to_sequence` |

**Note for the report:** ACF peaks at lags 268 and 335 are multiples of 67 (67 x 4, 67 x 5), suggesting a base cycle of about 67 points, which matches the ACF peak in 18.3. The loader uses 125, a fallback constant. The 496-point window still spans about 7 base cycles. This is an author preprocessing detail to report; keep it unchanged for the reproduction and test other periods only as an extension.

## 19.2. Windowing: verify with the repo code

Uses the repo's own `SequenceWindowConversion`. Expected for group 236: period 125 (find_length_rank falls back to its default 125; find_period = 335), `window_length=496, train_stride=6, test_stride=125`, test `pad=122, windows=(318, 496, 1)`, `round-trip OK=True`. Needs 17.2–17.3 and 18.1.


In [ ]:
from toolkit.utils import SequenceWindowConversion, find_length_rank, find_period

PATCH_SIZE, WINDOW_MULTIPLE = 8, 4          # values from runners/run_all_CoAD.sh

raw_first_col = pd.read_csv(f'dataset/TSB-AD/raw/{SOURCE_FILE}').dropna().iloc[:, 0]
print('find_length_rank:', find_length_rank(raw_first_col, rank=1),
      '| find_period:', find_period(raw_first_col), '| loader period:', periods[0])

# Same formulas as train_classify_mask.py (lines 84-105)
sub = int(periods[0])
multiple = WINDOW_MULTIPLE if len(train) // sub > WINDOW_MULTIPLE else len(train) // sub
window_length = (sub * multiple) // PATCH_SIZE * PATCH_SIZE
sub = min(sub, window_length)
n = len(train)
train_stride = 2 if n < 1000 else 6 if n < 10000 else 10 if n < 60000 else PATCH_SIZE if n < 100000 else sub
print(f'window_length={window_length}, train_stride={train_stride}, test_stride={sub}')

for name, data, stride in [('train', train, train_stride), ('test', test, sub)]:
    conv = SequenceWindowConversion(window_size=window_length, stride_size=stride)
    windows = conv.sequence_to_windows(as_2d(data))
    back = conv.windows_to_sequence(windows)
    print(f'{name}: length={len(data)}, pad={conv.pad_length}, windows={windows.shape}, '
          f'round-trip OK={back.shape[0] == len(data) and np.allclose(back[:, 0], data)}')

# How many windows cover each test point (overlap used for averaging scores)
conv = SequenceWindowConversion(window_size=window_length, stride_size=sub)
conv.sequence_to_windows(as_2d(test))
cover = np.zeros(len(conv.pad_sequence_data))
for i in range(len(conv.windows)):
    cover[i * sub: i * sub + window_length] += 1
cover = cover[conv.pad_length:]
print('Windows covering each test point: min', int(cover.min()), '| median', int(np.median(cover)),
      '| max', int(cover.max()))

for start in range(0, 3 * sub, sub):        # first three test windows, in padded coordinates
    print(f'Test window [{start - conv.pad_length}:{start - conv.pad_length + window_length}) '
          'in original indices (negative = zero padding)')

## 20.1. Run helper and metric summary

`run_logged` creates `runs/<UTC time>_<tag>/` on Drive with `run.json` (command, commit, status, seconds) and `console.log`. `summarize_metrics` flattens the evaluator JSON into 7 numbers: point-wise F1 (best threshold, uses test labels: optimistic), AUC-PR, AUC-ROC, and R_AUC_ROC, R_AUC_PR, VUS_ROC, VUS_PR. All in 0-1, higher is better; TSB-AD recommends VUS-PR as the main metric.


In [ ]:
import json, time, shutil
from datetime import datetime, timezone

def run_logged(cmd, tag):
    """Run one experiment command in the repo; keep command, status, timing and console log on Drive."""
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + tag
    out = STORE / 'runs' / run_id
    out.mkdir(parents=True, exist_ok=False)
    metadata = {'command': cmd, 'cwd': str(REPO), 'commit': commit, 'status': 'running'}
    (out / 'run.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    started = time.perf_counter()
    with (out / 'console.log').open('w', encoding='utf-8') as log:
        result = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    metadata.update(returncode=result.returncode, seconds=time.perf_counter() - started,
                    status='success' if result.returncode == 0 else 'failed')
    (out / 'run.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('Run:', out, 'status:', metadata['status'], f"({metadata['seconds']:.0f}s)")
    tail((out / 'console.log').read_text(encoding='utf-8'), 15 if result.returncode == 0 else 40)
    if result.returncode:
        raise RuntimeError('Run failed; inspect console.log before continuing')
    return out

def summarize_metrics(metric_file):
    """Flatten the evaluator JSON into the 7 headline numbers."""
    m = json.loads(Path(metric_file).read_text())
    row = {k: m['point_wise'][k] for k in ['f1_score', 'auc_prc', 'auc_roc']}
    row.update(m['vus'])
    return row

## 20.2. Baseline PCA on group 236

First cell declares the imports baseline_compare.py uses (`os`, `np`, `torch`) at the top of the script, because `toolkit.utils` exports only the names in its `__all__`; the 3-line change is saved as `env/baseline_compare_imports.diff`. Re-run it after every fresh clone. Second cell: one model x one group per call (no inner loop). Data-mining baselines fit directly on the test series. PCA is CPU-only and deterministic. The repo overwrites `baseline_result/` on every run, so outputs are copied to Drive right away. Expected: success, score length 39999.


In [ ]:
# Declare the imports baseline_compare.py uses (os, np, torch) at the top of the script:
# toolkit.utils exports only the names listed in its __all__. Record the change as a diff.
baseline_script = REPO / 'baseline_compare.py'
required_imports = 'import os\nimport numpy as np\nimport torch\n'
script_source = baseline_script.read_text(encoding='utf-8')
if not script_source.startswith(required_imports):
    baseline_script.write_text(required_imports + script_source, encoding='utf-8')
diff_text = subprocess.check_output(['git', 'diff', '--', 'baseline_compare.py'], cwd=REPO, text=True)
(STORE / 'env' / 'baseline_compare_imports.diff').write_text(diff_text, encoding='utf-8')
print(diff_text)

In [ ]:
BASELINE = 'PCA'
grep_file(REPO / 'runners' / 'run_dm_baseline.sh', rf'{BASELINE} .*TSB-AD')   # confirm the runner uses this model on TSB-AD

baseline_cmd = [sys.executable, 'baseline_compare.py',
                '--model_name', BASELINE, '--dataset_name', DATASET, '--group_name', GROUP]
baseline_run = run_logged(baseline_cmd, f'baseline_{BASELINE}_{DATASET}_{GROUP}')

# The repo overwrites baseline_result/ on every run: copy this run's outputs to Drive now.
src = REPO / 'baseline_result' / DATASET / BASELINE
artifacts = baseline_run / 'artifacts'
for rel in [f'data/group_{GROUP.zfill(3)}.json', f'anomaly_score/group_{GROUP.zfill(3)}.npy']:
    (artifacts / rel).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src / rel, artifacts / rel)

baseline_metrics = summarize_metrics(artifacts / f'data/group_{GROUP.zfill(3)}.json')
baseline_score = np.load(artifacts / f'anomaly_score/group_{GROUP.zfill(3)}.npy').reshape(-1)
print('Score length:', len(baseline_score), '| label length:', len(y),
      '| finite:', bool(np.isfinite(baseline_score).all()))
display(pd.DataFrame([baseline_metrics], index=[BASELINE]).round(4))

## 20.3. CoAD run

Same flags as `runners/run_all_CoAD.sh`. Set `NUM_EPOCHS = 3` for a quick pipeline test, then `NUM_EPOCHS = 300` (the runner setting) for the run used in comparisons; the run name, table label and report file names all include the epoch count. Files are found with `glob` (actual name: `group_236.json`). Needs a GPU runtime. Observed on group 236: 300 epochs took 184 s, VUS-PR 0.945.


In [ ]:
import torch
grep_file(REPO / 'runners' / 'run_all_CoAD.sh', r'TSB-AD')
assert torch.cuda.is_available(), 'Select a GPU runtime first'

NUM_EPOCHS = 300   # 3 = quick pipeline test; 300 = setting used in runners/run_all_CoAD.sh
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
coad_cmd = [
    sys.executable, 'train_classify_mask.py',
    '--dataset_name', DATASET, '--group_name', GROUP,
    '--model_name', 'freq_tf_v2', '--timestamp', timestamp,
    '--device', 'cuda:0', '--plot', 'False',
    '--num_epochs', str(NUM_EPOCHS),
    '--batch_size', '64' if DATASET == 'UCR' else '128',
    '--window_multiple', '4', '--lr', '2e-3',
    '--patch_size', '8', '--omega', '10', '--random_seed', '46',
]
coad_label = f'CoAD ({NUM_EPOCHS} epochs)'
coad_run = run_logged(coad_cmd, f'coad_{NUM_EPOCHS}ep_{DATASET}_{GROUP}')

artifact_dir = REPO / 'mask_results' / DATASET / f'freq_tf_v2_max_{timestamp}'
shutil.copytree(artifact_dir, coad_run / 'artifacts')
metric_files = sorted((coad_run / 'artifacts' / 'all' / 'data').glob('group_*.json'))
assert len(metric_files) == 1, f'Expected one metric file, found {metric_files}'
print('Metric file:', metric_files[0].name)
coad_metrics = summarize_metrics(metric_files[0])
display(pd.DataFrame([coad_metrics], index=[coad_label]).round(4))

## 20.4. Comparison table and score plot

Last row = CoAD - baseline (positive = CoAD better). Report files are named `comparison_<dataset>_<group>_<baseline>_vs_coad<epochs>ep.csv`. One series, one seed: Paper value is N/A at this scope. CoAD sets the first 125 score points to 0 by design.


In [ ]:
comparison = pd.DataFrame([baseline_metrics, coad_metrics],
                          index=[f'{BASELINE} (baseline)', coad_label])
comparison.loc['CoAD - baseline'] = comparison.iloc[1] - comparison.iloc[0]
report_tag = f'{DATASET}_{GROUP}_{BASELINE}_vs_coad{NUM_EPOCHS}ep'
comparison.to_csv(STORE / 'reports' / f'comparison_{report_tag}.csv')
display(comparison.round(4))

coad_score_file = next((coad_run / 'artifacts' / 'anomaly_score').glob('group_*_test_anomaly_score.npy'))
coad_score = np.load(coad_score_file).reshape(-1)
fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
ax[0].plot(as_2d(test)[:, 0], linewidth=0.6); ax[0].set_ylabel('Signal')
ax[1].plot(baseline_score, linewidth=0.6); ax[1].set_ylabel(f'{BASELINE} score')
ax[2].plot(coad_score, linewidth=0.6); ax[2].set_ylabel(f'{coad_label} score')
for axis in ax:
    for start, end in anomaly_segments(y):
        axis.axvspan(start, end, color='red', alpha=0.2)
ax[2].set_xlabel('Test sample index')
fig.tight_layout()
fig.savefig(STORE / 'reports' / f'scores_{report_tag}.png', dpi=160)
plt.show()

## 21.1. Load the score and check alignment

Score length must equal label length; never truncate labels to match.


In [ ]:
# Find the score file by pattern (actual name: group_236_test_anomaly_score.npy)
score_file = next((coad_run / 'artifacts' / 'anomaly_score').glob('group_*_test_anomaly_score.npy'))
score = np.load(score_file).reshape(-1)
assert len(score) == len(y), 'Do not truncate labels to match scores; inspect the mapping in the repository'
assert np.isfinite(score).all(), 'Scores contain NaN/Inf'
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax[0].plot(b[:, 0], linewidth=0.7)
ax[1].plot(score, linewidth=0.7)
for axis in ax:
    for start, end in anomaly_segments(y):
        axis.axvspan(start, end, color='red', alpha=0.2)
ax[0].set_ylabel('Signal'); ax[1].set_ylabel('Anomaly score')
ax[1].set_xlabel('Test sample index')
fig.tight_layout()
fig.savefig(coad_run / 'signal_score.png', dpi=160)
plt.show()